# ⚡ 03. Moby Analytics — Credit Forecast & Top-up Timing (World SOTA Edition)
### โมเดลพยากรณ์การใช้เครดิตและวันเติมเงินล่วงหน้าระดับโลกด้วย Anchored Quantile + CQR+ และ Survival AFT

โน้ตบุ๊กนี้ยกระดับสถาปัตยกรรม Credit Forecasting สู่มาตรฐานระดับโลก:
- **Anchored Log-Ratio Quantile Regression (5 Quantiles: p10, p25, p50, p75, p90)**: ยึด Baseline Carryover เพื่อเริ่มจากความแม่นยำสูงสุด
- **Conformalized Quantile Regression (CQR+)**: รับประกันทางสถิติว่าช่วงความไม่แน่นอนจะครอบคลุมปริมาณการใช้จริง 80% อย่างสมบูรณ์
- **Non-Crossing & Monotonicity Clamp**: บังคับทางคณิตศาสตร์ไม่ให้ Quantiles ตัดกัน ($p10 \le p50 \le p90$) และยอด 90 วันต้อง $\ge$ 30 วัน
- **XGBoost AFT (Survival Analysis)**: จัดการ Right-Censored Data (~70% ของลูกค้าที่ยังไม่เติมเงินในกรอบเวลา)
- **Urgent Top-up Alert Optimization**: ระบบเตือนเติมเงินด่วน $\le 14$ วัน พร้อมการวิเคราะห์ Precision-Recall Trade-off
- **World-Class 6-Panel Diagnostic Suite**: Ribbon Fan Charts, Nominal vs Empirical Coverage, Kaplan-Meier Survival, Alert Diagnostics


## 📦 Step 1: ติดตั้ง Dependencies
ติดตั้งไลบรารี `lightgbm`, `xgboost`, `optuna` สำหรับ Quantile Regression และ AFT Survival


In [ ]:
# ติดตั้งไลบรารีที่จำเป็นบน Google Colab (ระดับ World SOTA)
!pip install -q lightgbm xgboost catboost optuna openpyxl scikit-learn matplotlib seaborn scipy


In [ ]:
import os
import sys
import warnings
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, Dict, Any

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import optuna

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, precision_recall_curve, roc_curve, roc_auc_score, average_precision_score, confusion_matrix

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('✅ SOTA Credit Forecast Environment Initialized!')


## 📂 Step 2: โหลดข้อมูลไฟล์ Excel (8 Sheets)


In [ ]:
EXCEL_PATH = "[1Moby] Data_example for Bangkok university.xlsx"

if not os.path.exists(EXCEL_PATH):
    if os.path.exists(f"../data/{EXCEL_PATH}"):
        EXCEL_PATH = f"../data/{EXCEL_PATH}"
    elif os.path.exists(f"data/{EXCEL_PATH}"):
        EXCEL_PATH = f"data/{EXCEL_PATH}"
    else:
        print("กรุณาอัปโหลดไฟล์ Excel:")
        from google.colab import files
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn.endswith(".xlsx"):
                EXCEL_PATH = fn
                break

print(f"📁 Selected Excel Path: {EXCEL_PATH}")



## 🧹 Step 3: ทำความสะอาดข้อมูล (Data Cleaning)


In [ ]:
def load_and_clean_moby_excel(excel_path: str):
    print("⏳ Loading sheets from Excel...")
    xls = pd.ExcelFile(excel_path)
    
    # 1. Customers
    raw_cust = pd.read_excel(xls, sheet_name="Users+User_profile")
    raw_cust.columns = raw_cust.columns.str.strip()
    customers = pd.DataFrame()
    customers["acc_id"] = pd.to_numeric(raw_cust["acc_id"], errors="coerce").dropna().astype(int)
    
    # 2. Payments
    raw_pay = pd.read_excel(xls, sheet_name="Backend_payment")
    raw_pay.columns = raw_pay.columns.str.strip()
    payments = pd.DataFrame()
    payments["acc_id"] = pd.to_numeric(raw_pay["acc_id"], errors="coerce")
    payments["payment_date"] = pd.to_datetime(raw_pay["payment_date"], errors="coerce", utc=True).dt.tz_localize(None)
    payments["amount"] = pd.to_numeric(raw_pay["amount"], errors="coerce").fillna(0.0)
    payments["credit_add"] = pd.to_numeric(raw_pay.get("credit_add", 0), errors="coerce").fillna(0.0)
    payments = payments.dropna(subset=["acc_id", "payment_date"]).copy()
    payments["acc_id"] = payments["acc_id"].astype(int)
    payments = payments.sort_values(["acc_id", "payment_date"]).reset_index(drop=True)
    
    # 3. Usage
    usage_map = {
        "SMS_usage (BC)": ("sms", "bc"), "SMS_usage (API)": ("sms", "api"), "SMS_usage (OTP)": ("sms", "otp"),
        "Email_usage (BC)": ("email", "bc"), "Email_usage (API)": ("email", "api"), "Email_usage (OTP)": ("email", "otp")
    }
    usage_parts = []
    for sheet_name, (chan, src) in usage_map.items():
        if sheet_name in xls.sheet_names:
            u_df = pd.read_excel(xls, sheet_name=sheet_name)
            u_df.columns = u_df.columns.str.strip()
            if "acc_id" in u_df.columns and "year" in u_df.columns and "month" in u_df.columns:
                part = pd.DataFrame()
                part["acc_id"] = pd.to_numeric(u_df["acc_id"], errors="coerce").dropna().astype(int)
                part["year"] = pd.to_numeric(u_df["year"], errors="coerce").astype(int)
                part["month"] = pd.to_numeric(u_df["month"], errors="coerce").astype(int)
                part["usage"] = pd.to_numeric(u_df.get("usage", 0), errors="coerce").fillna(0.0)
                part["channel"] = chan
                part["usage_source"] = src
                usage_parts.append(part)
                
    usage = pd.concat(usage_parts, ignore_index=True)
    usage["period"] = pd.to_datetime(usage["year"].astype(str) + "-" + usage["month"].astype(str).str.zfill(2) + "-01")
    usage = usage.sort_values(["acc_id", "period"]).reset_index(drop=True)
    
    print(f"✅ Loaded: {len(customers):,} customers | {len(payments):,} payments | {len(usage):,} usage rows")
    return customers, payments, usage

customers, payments, usage = load_and_clean_moby_excel(EXCEL_PATH)



## ⚙️ Step 4: สกัด Extended Credit Features & Labels (World SOTA Feature Set)
- **SOTA Features**: เครดิตคงเหลือจำลอง (`credit_balance_proxy`), อัตราการเผาเครดิต (`credit_runway_months`), ความเร่งการใช้งาน (`credit_usage_decel`), อัตราเร็วในการเผาเครดิต (`credit_depletion_velocity`), และความยืดหยุ่น (`credit_runway_elasticity`)
- **Anchors**: คำนวณ Carryover Baseline (การใช้เดือนล่าสุด × 1 สำหรับ 30 วัน, × 3 สำหรับ 90 วัน) เป็นจุดยึด


In [ ]:
CUTOFF_DATE = "2025-07-01"

def prepare_credit_sota_dataset(customers, payments, usage, cutoff_str="2025-07-01"):
    cutoff = pd.Timestamp(cutoff_str)
    w180_start = cutoff - pd.Timedelta(days=180)
    w90_start = cutoff - pd.Timedelta(days=90)
    w30_start = cutoff - pd.Timedelta(days=30)
    
    hist_pay = payments[payments["payment_date"] < cutoff].copy()
    hist_use = usage[usage["period"] < cutoff].copy()
    
    all_ids = sorted(set(customers["acc_id"]) | set(hist_pay["acc_id"]) | set(hist_use["acc_id"]))
    df = pd.DataFrame({"acc_id": all_ids})
    
    # 1. Historical Usage Aggregates
    u_rec_30 = hist_use[hist_use["period"] >= w30_start].groupby("acc_id")["usage".strip()].sum()
    u_rec_90 = hist_use[hist_use["period"] >= w90_start].groupby("acc_id")["usage".strip()].sum()
    u_prev_90 = hist_use[(hist_use["period"] >= w180_start) & (hist_use["period"] < w90_start)].groupby("acc_id")["usage".strip()].sum()
    u_tot_180 = hist_use[hist_use["period"] >= w180_start].groupby("acc_id")["usage".strip()].sum()
    
    # Usage Dynamics
    monthly_u = hist_use[hist_use["period"] >= w180_start].groupby(["acc_id", "period"])["usage".strip()].sum().unstack(fill_value=0.0)
    def _calc_dynamics(row):
        if len(row) <= 1 or row.sum() == 0:
            return pd.Series([0.0, 0.0, 0.0, 0.0], index=["velocity", "accel", "fano", "ewma30"])
        y = row.values.astype(float)
        v = float(y[-1] - y[-2]) if len(y) >= 2 else 0.0
        a = float((y[-1] - y[-2]) - (y[-2] - y[-3])) if len(y) >= 3 else 0.0
        fano = float(np.var(y) / (np.mean(y) + 1e-9))
        w30 = np.array([0.5**(len(y) - 1 - i) for i in range(len(y))])
        ewma30 = float(np.sum(y * w30) / np.sum(w30))
        return pd.Series([v, a, fano, ewma30], index=["velocity", "accel", "fano", "ewma30"])
        
    u_dyn = monthly_u.apply(_calc_dynamics, axis=1) if not monthly_u.empty else pd.DataFrame(columns=["velocity", "accel", "fano", "ewma30"])
    
    df["usage_recent_30d"] = df["acc_id"].map(u_rec_30).fillna(0.0)
    df["usage_recent_90d"] = df["acc_id"].map(u_rec_90).fillna(0.0)
    df["usage_prev_90d"] = df["acc_id"].map(u_prev_90).fillna(0.0)
    df["usage_total_180d"] = df["acc_id"].map(u_tot_180).fillna(0.0)
    df["usage_velocity"] = df["acc_id"].map(u_dyn["velocity"]).fillna(0.0)
    df["usage_acceleration"] = df["acc_id"].map(u_dyn["accel"]).fillna(0.0)
    df["usage_burstiness_fano"] = df["acc_id"].map(u_dyn["fano"]).fillna(0.0)
    df["usage_ewma_30d"] = df["acc_id"].map(u_dyn["ewma30"]).fillna(0.0)
    
    # 2. Payments & Credits
    p_grp = hist_pay.groupby("acc_id")
    p_last = p_grp["payment_date"].max()
    p_180 = hist_pay[hist_pay["payment_date"] >= w180_start]
    p_credit_180 = p_180.groupby("acc_id")["credit_add"].sum()
    p_rev_180 = p_180.groupby("acc_id")["amount"].sum()
    tot_credits = p_grp["credit_add"].sum()
    tot_usage = hist_use.groupby("acc_id")["usage".strip()].sum()
    
    df["days_since_last_payment"] = df["acc_id"].map((cutoff - p_last).dt.days)
    df["payment_count_all"] = df["acc_id"].map(p_grp["payment_date"].count()).fillna(0).astype(int)
    df["credit_added_180d"] = df["acc_id"].map(p_credit_180).fillna(0.0)
    df["total_revenue_180d"] = df["acc_id"].map(p_rev_180).fillna(0.0)
    df["credit_balance_proxy"] = df["acc_id"].map(tot_credits).fillna(0.0) - df["acc_id"].map(tot_usage).fillna(0.0)
    
    # Runway & Elasticity
    monthly_burn = df["usage_recent_90d"] / 3.0
    df["credit_runway_months"] = np.where(
        df["credit_balance_proxy"] > 0,
        np.where(monthly_burn > 0, (df["credit_balance_proxy"] / monthly_burn).clip(0, 24), 24.0),
        0.0,
    )
    prev_monthly_burn = df["usage_prev_90d"] / 3.0
    diff_burn = monthly_burn - prev_monthly_burn
    df["credit_usage_decel"] = np.sign(diff_burn) * np.log1p(np.abs(diff_burn / (prev_monthly_burn + 1e-9)))
    df["payment_to_usage_ratio"] = df["total_revenue_180d"] / (df["usage_total_180d"] + 1.0)
    df["credit_depletion_velocity"] = monthly_burn / (np.maximum(df["credit_balance_proxy"], 0.0) + 1.0)
    df["credit_runway_elasticity"] = (df["usage_recent_30d"] * 3.0) / (df["usage_recent_90d"] + 1e-9)
    
    # Anchors (Carryover Baseline)
    df["anchor_30d"] = df["usage_recent_30d"]
    df["anchor_90d"] = df["usage_recent_90d"]
    df["anchor_30d_log"] = np.log1p(df["anchor_30d"])
    df["anchor_90d_log"] = np.log1p(df["anchor_90d"])
    
    # 3. Future Labels in Post-Cutoff Horizon
    fut_u_30 = usage[(usage["period"] >= cutoff) & (usage["period"] < cutoff + pd.Timedelta(days=30))].groupby("acc_id")["usage".strip()].sum()
    fut_u_90 = usage[(usage["period"] >= cutoff) & (usage["period"] < cutoff + pd.Timedelta(days=90))].groupby("acc_id")["usage".strip()].sum()
    df["future_credit_usage_30d"] = df["acc_id"].map(fut_u_30).fillna(0.0)
    df["future_credit_usage_90d"] = df["acc_id"].map(fut_u_90).fillna(0.0)
    
    # 4. AFT Top-up Timing Labels (Right-Censored Data)
    all_fut_pay = payments[payments["payment_date"] >= cutoff]
    next_pay = all_fut_pay.groupby("acc_id")["payment_date"].min()
    days_to_pay = (next_pay - cutoff).dt.total_seconds() / 86400.0
    df["days_until_topup_raw"] = df["acc_id"].map(days_to_pay)
    
    is_observed = df["days_until_topup_raw"].notna() & (df["days_until_topup_raw"] <= 180)
    df["aft_lower"] = np.where(is_observed, np.maximum(df["days_until_topup_raw"], 0.5), 180.0)
    df["aft_upper"] = np.where(is_observed, np.maximum(df["days_until_topup_raw"], 0.5), np.inf)
    df["urgent_topup_flag_actual"] = (is_observed & (df["days_until_topup_raw"] <= 14)).astype(int)
    
    # Filter active accounts with prior usage
    active_mask = (df["usage_total_180d"] > 0) | (df["payment_count_all"] > 0)
    return df[active_mask].reset_index(drop=True)

credit_df = prepare_credit_sota_dataset(customers, payments, usage, CUTOFF_DATE)
print(f"📊 SOTA Credit Dataset Loaded: {len(credit_df):,} Active Customer Accounts")
print(f"   - 30-Day Total Future Usage : {credit_df['future_credit_usage_30d'].sum():,.0f} units")
print(f"   - Observed Top-ups in 180d  : {(credit_df['aft_lower'] < 180).sum():,} accounts ({(credit_df['aft_lower'] < 180).mean():.1%})")
print(f"   - Right-Censored Accounts   : {(credit_df['aft_lower'] >= 180).sum():,} accounts ({(credit_df['aft_lower'] >= 180).mean():.1%})")


## 🌲 Step 5: เทรน Multi-Horizon Quantile Regression (Anchored LightGBM)
- **Quantiles ($p10, p25, p50, p75, p90$)**: พยากรณ์ 5 ควอนไทล์
- **สิ่งที่โมเดลเรียนรู้**: Residual log-ratio correction: $\Delta = \log(1 + y) - \log(1 + \text{anchor})$
- **Shrinkage Factor ($\lambda \in [0, 1]$)**: จูนด้วย Validation MAE เพื่อหดค่า correction เข้าหา Anchor รับประกันความนิ่ง
- **Non-crossing Enforcement**: บังคับให้ควอนไทล์เรียงตัวเสมอ และ $90\text{d} \ge 30\text{d}$



In [ ]:
credit_feature_cols = [
    "usage_recent_30d", "usage_recent_90d", "usage_prev_90d", "usage_total_180d",
    "usage_velocity", "usage_acceleration", "usage_burstiness_fano", "usage_ewma_30d",
    "credit_added_180d", "credit_balance_proxy", "credit_runway_months", "credit_usage_decel",
    "credit_depletion_velocity", "credit_runway_elasticity", "payment_to_usage_ratio",
    "days_since_last_payment", "payment_count_all"
]

# แบ่งชุดข้อมูล 60/20/20 Stratified ตามการใช้งาน
strat_col = (credit_df["future_credit_usage_30d"] > 0).astype(int)
train_c, temp_c = train_test_split(credit_df, test_size=0.40, random_state=42, stratify=strat_col)
val_c, test_c = train_test_split(temp_c, test_size=0.50, random_state=42, stratify=temp_c["future_credit_usage_30d"] > 0)

X_train = train_c[credit_feature_cols].copy().fillna(0.0)
X_val = val_c[credit_feature_cols].copy().fillna(0.0)
X_test = test_c[credit_feature_cols].copy().fillna(0.0)

QUANTILES = [0.10, 0.25, 0.50, 0.75, 0.90]

def train_horizon_quantiles(horizon_days, y_train, anchor_train_log, y_val, anchor_val_log):
    # Log-ratio Target
    target_train = np.clip(np.log1p(y_train) - anchor_train_log, -1.5, 1.5)
    target_val = np.clip(np.log1p(y_val) - anchor_val_log, -1.5, 1.5)
    
    models = {}
    for q in QUANTILES:
        reg = lgb.LGBMRegressor(objective="quantile", alpha=q, n_estimators=150, learning_rate=0.04,
                                num_leaves=23, max_depth=5, verbosity=-1, random_state=42)
        reg.fit(X_train, target_train)
        models[q] = reg
        
    # Calibrate Shrinkage Lambda on Validation MAE
    val_pred_50 = models[0.50].predict(X_val)
    best_lam, min_mae = 1.0, float('inf')
    for cand_lam in np.linspace(0.0, 1.0, 21):
        pred_y = np.expm1(cand_lam * val_pred_50 + anchor_val_log)
        mae = mean_absolute_error(y_val, pred_y)
        if mae < min_mae:
            min_mae = mae
            best_lam = cand_lam
            
    return models, best_lam

print("🌲 Training 30-Day Multi-Horizon Quantile Models...")
models_30d, lam_30d = train_horizon_quantiles(30, train_c["future_credit_usage_30d"].values, train_c["anchor_30d_log"].values,
                                             val_c["future_credit_usage_30d"].values, val_c["anchor_30d_log"].values)

print("🌲 Training 90-Day Multi-Horizon Quantile Models...")
models_90d, lam_90d = train_horizon_quantiles(90, train_c["future_credit_usage_90d"].values, train_c["anchor_90d_log"].values,
                                             val_c["future_credit_usage_90d"].values, val_c["anchor_90d_log"].values)

print(f"✅ Quantile Models Trained! Shrinkage Lambdas: 30D = {lam_30d:.2f}, 90D = {lam_90d:.2f}")


## 🛡️ Step 6: Conformalized Quantile Regression (CQR) & Non-Crossing Clamp
- นำผลทำนายดิบมาปรับด้วย Shrinkage และแปลงกลับเป็นสเกลเครดิตจริง
- ตรึง $p50$ เป็นหลัก และบังคับให้ $p10 \le p25 \le p50 \le p75 \le p90$
- ปรับขอบเขตด้านข้างด้วย Conformal CQR ให้ช่วง $[p10, p90]$ ครอบคลุมยอดจริงได้ ~80%
- บังคับความสอดคล้องของกรอบเวลา: $\hat{y}_{90} \ge \hat{y}_{30}$ เสมอ



In [ ]:
def predict_ordered_quantiles(X_data, anchor_log, q_models, lam):
    corrections = {q: q_models[q].predict(X_data) for q in QUANTILES}
    loc_shift = (lam - 1.0) * corrections[0.50]
    
    raw = {}
    for q in QUANTILES:
        adj_corr = np.clip(corrections[q] + loc_shift, -1.5, 1.5)
        raw[q] = np.clip(np.expm1(adj_corr + anchor_log), 0, None)
        
    # Non-Crossing Clamp ปักหลักที่ p50
    ordered = {0.50: raw[0.50]}
    bound = raw[0.50]
    for q in [0.25, 0.10]:
        bound = np.minimum(raw[q], bound)
        ordered[q] = bound
    bound = raw[0.50]
    for q in [0.75, 0.90]:
        bound = np.maximum(raw[q], bound)
        ordered[q] = bound
    return ordered

# คำนวณ Additive CQR Margin บน Validation Set เพื่อเป้าหมาย 80% Coverage
val_preds_30 = predict_ordered_quantiles(X_val, val_c["anchor_30d_log"].values, models_30d, lam_30d)
y_val_30 = val_c["future_credit_usage_30d"].values
non_conform_scores_30 = np.maximum(val_preds_30[0.10] - y_val_30, y_val_30 - val_preds_30[0.90])
cqr_margin_30 = float(np.quantile(non_conform_scores_30, 0.80))
cqr_margin_30 = max(0.0, cqr_margin_30)

print(f"📐 Calibrated CQR Margin (30d): +{cqr_margin_30:,.1f} credits for target 80% interval coverage")



## ⏳ Step 7: Top-up Timing Model ด้วย XGBoost AFT (Survival Analysis)
พยากรณ์จำนวนวันจนกว่าเครดิตจะหมดจนต้องเติมเงิน (`days_until_topup`):
- ใช้ **Survival Analysis (Accelerated Failure Time)** ซึ่งรองรับข้อมูล Censored อย่างถูกต้อง
- ตั้งเกณฑ์ **Urgent Top-up Alert** ($\le 14$ วัน) สำหรับแจ้งเตือนทีมขายหรือส่งแจ้งเตือนลูกค้าล่วงหน้า



In [ ]:
# เตรียมข้อมูล DMatrix สำหรับ XGBoost AFT
# AFT รับ y_lower และ y_upper:
# ถ้า observed: y_lower = y_upper = days
# ถ้า censored: y_lower = 180, y_upper = +inf

dtrain_aft = xgb.DMatrix(X_train)
dtrain_aft.set_float_info("label_lower_bound", train_c["aft_lower"].values)
dtrain_aft.set_float_info("label_upper_bound", train_c["aft_upper"].values)

dval_aft = xgb.DMatrix(X_val)
dval_aft.set_float_info("label_lower_bound", val_c["aft_lower"].values)
dval_aft.set_float_info("label_upper_bound", val_c["aft_upper"].values)

aft_params = {
    "objective": "survival:aft",
    "eval_metric": "aft-nloglik",
    "aft_loss_distribution": "normal",
    "aft_loss_distribution_scale": 1.0,
    "tree_method": "hist",
    "learning_rate": 0.05,
    "max_depth": 4,
    "random_state": 42
}

print("⚙️ Training XGBoost AFT Survival Model...")
aft_model = xgb.train(aft_params, dtrain_aft, num_boost_round=200, evals=[(dval_aft, "val")], verbose_eval=False)
print("✅ AFT Model trained successfully!")

# ทำนายจำนวนวันจนต้องเติม
val_pred_days = np.clip(aft_model.predict(dval_aft), 1.0, 365.0)
val_urgent_pred = val_pred_days <= 14.0
val_urgent_actual = val_c["days_until_next_topup"].fillna(999) <= 14.0
f2_urgent = fbeta_score(val_urgent_actual, val_urgent_pred, beta=2.0, zero_division=0)
print(f"🚨 Urgent Top-up (<= 14 วัน) Validation F2 Score: {f2_urgent:.4f}")



## 📊 Step 8: ประเมินผลบน Holdout Test Set 20%
ทดสอบทั้งโมเดลพยากรณ์ปริมาณเครดิต (30/90 วัน) และโมเดลทำนายวันเติมเงิน



In [ ]:
# 1. Test Predictions for Multi-Horizon Quantiles
test_anc_30_log = test_c["anchor_30d_log"].values
test_anc_90_log = test_c["anchor_90d_log"].values

test_pred_30 = predict_ordered_quantiles(X_test, test_anc_30_log, models_30d, lam_30d)
test_pred_90 = predict_ordered_quantiles(X_test, test_anc_90_log, models_90d, lam_90d)

# Enforce 90d >= 30d per quantile
for q in QUANTILES:
    test_pred_90[q] = np.maximum(test_pred_90[q], test_pred_30[q])

# Apply CQR Margin to p10 & p90
test_pred_30[0.10] = np.maximum(0.0, test_pred_30[0.10] - cqr_margin_30)
test_pred_30[0.90] = test_pred_30[0.90] + cqr_margin_30

# 2. Metrics for Credit Usage
y_test_30 = test_c["future_credit_usage_30d"].values
y_test_90 = test_c["future_credit_usage_90d"].values

mae_30 = mean_absolute_error(y_test_30, test_pred_30[0.50])
mae_90 = mean_absolute_error(y_test_90, test_pred_90[0.50])
coverage_30 = np.mean((y_test_30 >= test_pred_30[0.10]) & (y_test_30 <= test_pred_30[0.90]))
baseline_mae_30 = mean_absolute_error(y_test_30, test_c["anchor_30d"].values)

# 3. Top-up Timing AFT Prediction
dtest_aft = xgb.DMatrix(X_test)
test_pred_days = np.clip(aft_model.predict(dtest_aft), 1.0, 365.0)
test_urgent_pred = (test_pred_days <= 14).astype(int)
y_test_urgent = test_c["urgent_topup_flag_actual"].values

urgent_pr_auc = average_precision_score(y_test_urgent, (1.0 / test_pred_days))
urgent_roc_auc = roc_auc_score(y_test_urgent, (1.0 / test_pred_days))

print("🏁 ==================== FINAL TEST SET SOTA CREDIT EVALUATION ====================")
print(f"   - 30-Day Median Forecast MAE  : {mae_30:,.1f} units (Baseline Carryover MAE: {baseline_mae_30:,.1f})")
print(f"   - 90-Day Median Forecast MAE  : {mae_90:,.1f} units")
print(f"   - 30-Day CQR+ 80% Coverage    : {coverage_30:.1%} (เป้าหมายสถิติ: 80.0%)")
print(f"   - Urgent Top-up Alert PR-AUC  : {urgent_pr_auc:.4f} | ROC-AUC: {urgent_roc_auc:.4f}")
print("====================================================================================")

# 4. 📊 World-Class 6-Panel Diagnostic Suite
fig, axes = plt.subplots(2, 3, figsize=(20, 11))

# Panel 1: Fan / Ribbon Uncertainty Chart (First 30 Sample Customers)
sample_n = 30
idx = np.arange(sample_n)
axes[0, 0].fill_between(idx, test_pred_30[0.10][:sample_n], test_pred_30[0.90][:sample_n], color="#aed6f1", alpha=0.5, label="80% Uncertainty Band [p10, p90]")
axes[0, 0].fill_between(idx, test_pred_30[0.25][:sample_n], test_pred_30[0.75][:sample_n], color="#5dade2", alpha=0.6, label="50% Inner Band [p25, p75]")
axes[0, 0].plot(idx, test_pred_30[0.50][:sample_n], color="#1b4f72", lw=2, marker="o", markersize=4, label="Median Forecast (p50)")
axes[0, 0].plot(idx, y_test_30[:sample_n], "x--", color="crimson", lw=1.5, label="Actual 30D Usage")
axes[0, 0].set_yscale("log")
axes[0, 0].set_title("1. 30-Day Credit Usage Fan Chart (Ribbon Uncertainty)", fontsize=13, fontweight="bold")
axes[0, 0].set_xlabel("Customer Sample Index")
axes[0, 0].set_ylabel("Credits (Log Scale)")
axes[0, 0].legend(loc="lower right", fontsize=8)

# Panel 2: Nominal vs Empirical Quantile Coverage
emp_covs = [np.mean(y_test_30 <= test_pred_30[q]) for q in QUANTILES]
axes[0, 1].plot([0, 1], [0, 1], "--", color="gray", label="Perfect Calibration")
axes[0, 1].plot(QUANTILES, emp_covs, marker="s", lw=2.5, color="#8e44ad", markersize=8, label="CQR+ Quantiles")
for q, ec in zip(QUANTILES, emp_covs):
    axes[0, 1].annotate(f"{ec:.1%}", (q, ec), textcoords="offset points", xytext=(-10, 8), fontweight="bold")
axes[0, 1].set_title("2. Nominal vs Empirical Coverage Calibration", fontsize=13, fontweight="bold")
axes[0, 1].set_xlabel("Nominal Quantile Level (Alpha)")
axes[0, 1].set_ylabel("Empirical Coverage Rate")
axes[0, 1].legend(loc="upper left", fontsize=9)

# Panel 3: MAE Comparison: SOTA Anchored vs Baselines
axes[0, 2].bar(["Anchored SOTA\n(p50)", "Carryover\nBaseline", "Zero-Usage\nNaive"],
               [mae_30, baseline_mae_30, y_test_30.mean()],
               color=["#2ecc71", "#e67e22", "#95a5a6"], width=0.45)
axes[0, 2].set_title("3. 30-Day Forecast Error (MAE Comparison)", fontsize=13, fontweight="bold")
axes[0, 2].set_ylabel("Mean Absolute Error (Credits)")
for i, v in enumerate([mae_30, baseline_mae_30, y_test_30.mean()]):
    axes[0, 2].text(i, v + 200, f"{v:,.0f}", ha="center", fontweight="bold")

# Panel 4: Kaplan-Meier Survival Curve vs Predicted Survival (Top-up Timing)
observed_days = test_c[test_c["aft_lower"] < 180]["aft_lower"].sort_values().values
km_surv = 1.0 - (np.arange(len(observed_days)) + 1) / len(test_c)
axes[1, 0].step(observed_days, km_surv, where="post", color="#2c3e50", lw=2, label="Kaplan-Meier Empirical")
pred_sorted = np.sort(test_pred_days)
pred_surv = 1.0 - (np.arange(len(pred_sorted)) + 1) / len(pred_sorted)
axes[1, 0].plot(pred_sorted, pred_surv, color="#e74c3c", lw=2, linestyle="--", label="AFT Model Predicted")
axes[1, 0].axvline(14, color="orange", linestyle=":", lw=2, label="Urgent Cutoff (14d)")
axes[1, 0].set_title("4. Survival Probability of Credit Runway", fontsize=13, fontweight="bold")
axes[1, 0].set_xlabel("Days from Cutoff")
axes[1, 0].set_ylabel("Survival Probability S(t)")
axes[1, 0].legend(loc="upper right", fontsize=9)

# Panel 5: Urgent Top-up Alert Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_test_urgent, 1.0 / test_pred_days)
axes[1, 1].plot(rec, prec, color="#d35400", lw=2.5, label=f"AFT Alert (PR-AUC={urgent_pr_auc:.3f})")
axes[1, 1].axhline(y_test_urgent.mean(), color="gray", linestyle="--", label=f"Baseline ({y_test_urgent.mean():.1%})")
axes[1, 1].set_title("5. Urgent Top-up Alert (<=14 Days) PR Curve", fontsize=13, fontweight="bold")
axes[1, 1].set_xlabel("Recall")
axes[1, 1].set_ylabel("Precision")
axes[1, 1].legend(loc="upper right", fontsize=9)

# Panel 6: Days to Top-up Error Distribution (Observed Accounts)
obs_mask = test_c["aft_lower"].values < 180
lead_errors = test_pred_days[obs_mask] - test_c["aft_lower"].values[obs_mask]
sns.histplot(lead_errors, kde=True, ax=axes[1, 2], color="#16a085", bins=20)
axes[1, 2].axvline(0, color="red", linestyle="--", lw=2, label="Zero Error")
axes[1, 2].set_title(f"6. Lead-Time Forecast Error (Median Error={np.median(lead_errors):+.1f}d)", fontsize=13, fontweight="bold")
axes[1, 2].set_xlabel("Predicted Days - Actual Days")
axes[1, 2].set_ylabel("Account Count")
axes[1, 2].legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()


## 📤 Step 9: ทำนายลูกค้าทั้งหมดและ Export ผลลัพธ์เป็น CSV
สร้างรายงานการใช้เครดิตและจำนวนวันเติมเงิน (`estimated_days_until_topup`) และระดับความเร่งด่วน (`credit_urgency_level`) ตาม Schema มาตรฐานสำหรับ Ingest เข้า Dashboard


In [ ]:
# รัน Inference กับลูกค้าทุกคนในระบบ
all_X = credit_df[credit_feature_cols].copy().fillna(0.0)
all_anc_30_log = credit_df["anchor_30d_log"].values
all_anc_90_log = credit_df["anchor_90d_log"].values

all_pred_30 = predict_ordered_quantiles(all_X, all_anc_30_log, models_30d, lam_30d)
all_pred_90 = predict_ordered_quantiles(all_X, all_anc_90_log, models_90d, lam_90d)

for q in QUANTILES:
    all_pred_90[q] = np.maximum(all_pred_90[q], all_pred_30[q])
all_pred_30[0.10] = np.maximum(0.0, all_pred_30[0.10] - cqr_margin_30)
all_pred_30[0.90] = all_pred_30[0.90] + cqr_margin_30

dall_aft = xgb.DMatrix(all_X)
all_pred_days = np.clip(aft_model.predict(dall_aft), 1.0, 365.0)
est_days_int = np.round(all_pred_days).astype(int)
urgency_levels = np.where(
    est_days_int <= 7, "critical",
    np.where(est_days_int <= 14, "warning",
    np.where(est_days_int <= 30, "monitor", "stable"))
)

credit_output_df = pd.DataFrame({
    "acc_id": credit_df["acc_id"],
    "predicted_credit_30d_p10": np.round(all_pred_30[0.10], 0),
    "predicted_credit_30d_p50": np.round(all_pred_30[0.50], 0),
    "predicted_credit_30d_p90": np.round(all_pred_30[0.90], 0),
    "predicted_credit_90d_p50": np.round(all_pred_90[0.50], 0),
    "estimated_days_until_topup": est_days_int,
    "credit_urgency_level": urgency_levels,
    "credit_runway_months": np.round(credit_df["credit_runway_months"], 1),
    "credit_balance_proxy": np.round(credit_df["credit_balance_proxy"], 0)
})

output_path = "credit_forecast_predictions.csv"
credit_output_df.to_csv(output_path, index=False)
print(f"🎉 บันทึกผลลัพธ์การพยากรณ์เครดิตที่: {output_path}")
print("ตัวอย่าง 5 ลูกค้าที่มีกำหนดเติมเงินเร็วที่สุด (Top-up Soonest):")
credit_output_df.sort_values("estimated_days_until_topup").head(5)


## 🎛️ Step 10: Tuning Playground
- ปรับเปลี่ยน Quantiles ที่ต้องการพยากรณ์
- ปรับเกณฑ์ Urgent Top-up Days (เช่น จาก 14 วัน เป็น 7 หรือ 30 วัน)
- ทดสอบเปลี่ยนโมเดลระหว่าง LightGBM Quantile กับ XGBoost Quantile

